In [ ]:
import pandas as pd
import requests

url = "https://data.cdc.gov/resource/i46a-9kgh.json"
params = {
    "$select": "stateabbr,countyname,countyfips,cholscreen_adjprev",
    "stateabbr": "CA"
}

r = requests.get(url, params=params)
r.raise_for_status()

chol_ca = pd.DataFrame(r.json())

chol_ca

In [ ]:
# read income data
income_data = pd.read_csv("../data/raw/income20261533.csv", skiprows=4)

income_data

# Merge datasets

In [ ]:
# Keep one income row per county before merging.
income_per_capita = income_data.loc[
    income_data["Income Type"].eq("Per Capita Personal Income - BEA")
].copy()

income_per_capita = income_per_capita.loc[
    income_per_capita["Area"].ne("California")
].copy()

income_per_capita["countyname"] = income_per_capita["Area"].str.replace(
    r"\s+County$", "", regex=True
).str.strip()
income_per_capita["per_capita_income"] = (
    income_per_capita["Income"]
    .replace({r"[$,]": ""}, regex=True)
    .astype(int)
)
income_per_capita["population"] = pd.to_numeric(
    income_per_capita["Population"], errors="coerce"
).astype("Int64")

income_per_capita = income_per_capita.rename(columns={"Year": "income_year"})[
    ["countyname", "income_year", "per_capita_income", "population"]
]

chol_clean = chol_ca.copy()
chol_clean["countyname"] = chol_clean["countyname"].str.strip()
chol_clean["countyfips"] = chol_clean["countyfips"].astype(str).str.zfill(5)
chol_clean["cholscreen_adjprev"] = pd.to_numeric(
    chol_clean["cholscreen_adjprev"], errors="coerce"
)

missing_from_income = sorted(
    set(chol_clean["countyname"]) - set(income_per_capita["countyname"])
)
missing_from_cholesterol = sorted(
    set(income_per_capita["countyname"]) - set(chol_clean["countyname"])
)

if missing_from_income or missing_from_cholesterol:
    raise ValueError(
        f"County mismatch. Missing income: {missing_from_income}; "
        f"missing cholesterol: {missing_from_cholesterol}"
    )

merged_data = (
    chol_clean.merge(
        income_per_capita,
        on="countyname",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("countyname")
    .reset_index(drop=True)
)

merged_data.to_csv("../data/processed/cholesterol_income_by_county.csv", index=False)

merged_data